In [1]:
from os import path
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm, LinearSegmentedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

In [8]:
def load_visibility_data(data_path='src/visibility_anisotropic_idw.nc'):
    """
    加载能见度数据
    """
    if not data_path.startswith('http') and not path.exists(data_path):
        print(f"错误：找不到文件 {data_path}")
        print("请确保已运行插值程序生成该文件")
        return None
    
    try:
        # 加载数据
        ds = xr.open_dataset(data_path)
        print(f"成功加载数据: {data_path}")
        print(f"数据维度: {ds.dims}")
        print(f"数据变量: {list(ds.data_vars)}")
        
        # 获取能见度数据
        if 'visibility' in ds:
            vis_data = ds['visibility']
        elif 'vis000' in ds:
            vis_data = ds['vis000'][0,0,:,:]
        else:
            # 尝试获取第一个数据变量
            var_name = list(ds.data_vars)[0]
            vis_data = ds[var_name]
            print(f"使用数据变量: {var_name}")
        
        # 转换单位如果需要（从m转换为km）
        if vis_data.max() > 100:  # 如果最大值大于100，可能是米单位
            vis_data = vis_data / 1000
            print("已将单位从米转换为千米")
        
        print(f"能见度数据范围: {vis_data.min().values:.2f} - {vis_data.max().values:.2f} km")
        
        return vis_data
        
    except Exception as e:
        print(f"加载数据时出错: {e}")
        return None

In [3]:
fields = [
    'V01301',    # 站号
    'VF01015_CN',# 站点名称
    'V_CITY',    # 所属地市
    'V_COUNTY',  # 所属县
    'V06001',    # 经度
    'V05001',    # 纬度
    'V07001',    # 海拔
    'V20001',    # 能见度
    'V13003'     # 相对湿度
]
df_nation = pd.read_csv('../data/SurfAuto_20250228000000.csv', encoding='gbk', na_values=9999, usecols=fields)
fields_2 = [
    'V01301',    # 站号
    'VF01015_CN',# 站点名称
    'V_CITY',    # 所属地市
    'V_COUNTY',  # 所属县
    'V06001',    # 经度
    'V05001',    # 纬度
    'V07001',    # 海拔
    'V13003',     # 相对湿度
    'V20001_701_01', # 能见度
]
df_region = pd.read_csv('../data/SurfAwst_20250228000000.csv', encoding='gbk', na_values=9999, usecols=fields_2)
# 重命名字段为英文名
field_map = {
    'V01301': 'code',
    'VF01015_CN': 'name',
    'V_CITY': 'city',
    'V_COUNTY': 'county',
    'V06001': 'lon',
    'V05001': 'lat',
    'V07001': 'altitude',
    'V20001': 'vis',
    'V13003': 'rh',
    'V20001_701_01': 'vis',
}

df_nation = df_nation.rename(columns=field_map)
df_region = df_region.rename(columns=field_map)
# 去除掉df_region中rh为NaN的条目
# df_region = df_region.dropna(subset=['rh', 'county'])
df_region = df_region.dropna(subset=['vis', 'county'])
print(df_nation.head())
print(df_region.count())

    code     lat      lon  altitude  rh   vis       name   county city
0  57988  25.110  113.345     143.2  96   800  乐昌国家基本气象站      乐昌市   韶关
1  57989  25.059  113.763     112.7  96   200  仁化国家基本气象站      仁化县   韶关
2  57996  25.081  114.255     149.7  95  1900  南雄国家基准气候站      南雄市   韶关
3  59071  24.732  112.278     174.3  94  2700  连南国家基本气象站  连南瑶族自治县   清远
4  59072  24.811  112.371     131.7  98   100     连州市气象局      连州市   清远
code        124
lon         124
lat         124
altitude    124
county      124
city        124
name        124
rh           88
vis         124
dtype: int64


In [5]:
df_region

,code,lon,lat,altitude,county,city,name,rh,vis
0,G8337,112.570939,24.482277,311.0,阳山县,清远,许广上行K1180 000M应用气象观测站（交通站）,NaN,1908.0
8,G1275,114.028900,22.095800,45.0,香洲区,珠海,香洲区担杆镇外伶仃流水坑气象观测站,NaN,9952.0
62,G8020,112.216100,23.815028,90.0,怀集县,肇庆,怀集怀城服务区气象观测站,NaN,2314.0
101,G5967,113.579400,22.863600,10.0,东莞市,东莞,东莞沙田镇作业区中路气象观测站,84.0,5109.0
109,G1238,113.421400,22.187500,0.0,香洲区,珠海,香洲区珠海大桥东气象观测站,84.0,9353.0
...,...,...,...,...,...,...,...,...,...
3835,G1281,113.818600,22.158100,7.0,香洲区,珠海,香洲区桂山镇桂山电厂气象观测站,92.0,31331.0
3843,G9576,113.233056,23.117778,8.6,荔湾区,广州,荔湾区多宝街永庆坊,84.0,7166.0
3871,G3133,113.489400,22.907200,12.6,番禺区,广州,番禺区石楼镇清流村,NaN,7116.0
4133,G9533,113.573097,22.816894,4.2,南沙区,广州,南沙区南沙街燃料港口气象观测站,89.0,8758.0


In [11]:
vis_data = load_visibility_data('http://10.148.8.71:7080/thredds/dodsC/cldas/20250228/VIS_2025022805.NC')

成功加载数据: http://10.148.8.71:7080/thredds/dodsC/cldas/20250228/VIS_2025022805.NC
数据维度: FrozenMappingWarningOnValuesAccess({'time': 1, 'level': 1, 'lat': 1201, 'lon': 1401})
数据变量: ['tstr', 'vis000']
已将单位从米转换为千米
能见度数据范围: -0.00 - 68.76 km


In [12]:
print(vis_data)

<xarray.DataArray 'vis000' (lat: 1201, lon: 1401)> Size: 7MB
array([[-0.001, -0.001, -0.001, ..., -0.001, -0.001, -0.001],
       [-0.001, -0.001, -0.001, ..., -0.001, -0.001, -0.001],
       [-0.001, -0.001, -0.001, ..., -0.001, -0.001, -0.001],
       ...,
       [-0.001, -0.001, -0.001, ..., -0.001, -0.001, -0.001],
       [-0.001, -0.001, -0.001, ..., -0.001, -0.001, -0.001],
       [-0.001, -0.001, -0.001, ..., -0.001, -0.001, -0.001]],
      shape=(1201, 1401), dtype=float32)
Coordinates:
  * lon      (lon) float32 6kB 70.0 70.05 70.1 70.15 ... 139.9 139.9 139.9 140.0
  * lat      (lat) float32 5kB 0.0 0.05 0.1 0.15 0.2 ... 59.85 59.9 59.95 60.0
    level    float32 4B 1e+03
    time     datetime64[ns] 8B 2025-02-28T05:00:00
